# Part I on larger-basis windows: H₂O and N₂ in cc-pVDZ

This notebook repeats the Part I measurement comparison (M1, M2, M3, M3-R) on the cc-pVDZ retained-orbital windows
from the archive notes, and adds two things that the new systems need:

* **Δ-good costs** (Δ = 1 mHa). In these systems the leading gradients are often exact ties (N₂ π partners) or
  closer than 1 mHa (H₂O at equilibrium, CISD). Exact-winner costs then blow up, so we also report the cost of
  finding a generator within Δ of the best, as Part II does.
* **M3-C**, an oracle contrast rule on the same parent groups. It uses the Part II idea (one interval on the
  difference |g_w| − |g_j|, with the covariance of the shared groups) but no coefficient splitting and no finite shots.

| System | Window | Qubits | Electrons | Geometries |
|---|---|---|---|---|
| H₂O | 8o: {2a1, 1b1} + CAS(4,4) + {3b2, 5a1}, 1a1 frozen | 16 | 8 | R(OH) = 0.9584, 1.75 Å, angle 104.45° |
| N₂ | 11o: {2σg, 2σu} + CAS(6,6) + {4ag, 2b2u, 2b3u}, 1s frozen | 22 | 10 | R = 1.10, 1.80, 3.00 Å |

The window RHF and window-FCI energies reproduce the archive table to 0.5 μHa at all five geometries (checked below).
H₂O at 3.0 Å is left out: its archive RHF solution (−75.430561 Ha) could not be reproduced; PySCF finds several other
RHF solutions there.

**Pools.** `full` = the spin-orbital UCCSD pool, as in Part I. `sym` = only the totally symmetric generators (C2v / D2h).
The others have exactly zero gradient in every symmetric state, so dropping them never changes the selection.
N₂ was run with `sym` only (the full pool has 1,260 generators and 1.65 M Pauli terms, too large for this 2-core machine).

Code is written to files by the cells below; the heavy runs are switched off (`RUN = False`) and the tables are read
from `results/partI/`.

### `windows.py` — Window construction and validation

In [1]:
%%writefile windows.py
"""cc-pVDZ retained-orbital windows for H2O (8o) and N2 (11o), following the archive notes.

Orbitals are chosen by irrep at the first geometry and traced to later geometries by
irrep-resolved maximum overlap (archive Sec. 3.3). The window Hamiltonian is the CASCI
effective Hamiltonian of the retained orbitals, so window FCI = CASCI over the window.
"""
import numpy as np
from pyscf import gto, scf, mcscf, fci, ao2mo


def water(r, angle=104.45):
    t = np.deg2rad(angle / 2)
    return f"O 0 0 0; H 0 {r*np.sin(t):.8f} {r*np.cos(t):.8f}; H 0 {-r*np.sin(t):.8f} {r*np.cos(t):.8f}"


def n2(r):
    return f"N 0 0 0; N 0 0 {r}"


def run_rhf(atom, group=True, guess="minao", newton=False):
    """RHF. The archive's stretched N2 solutions are reproduced by a core (1e) guess with
    second-order SCF; H2O uses the default guess."""
    mol = gto.M(atom=atom, basis="cc-pvdz", symmetry=group, unit="Angstrom", verbose=0)
    mf = scf.RHF(mol)
    mf.conv_tol = 1e-11
    mf.max_cycle = 300
    mf.init_guess = guess
    if newton:
        mf = mf.newton()
    mf.kernel()
    assert mf.converged
    return mol, mf


def labels(mol, mf):
    return list(scf.hf_symm.get_orbsym(mol, mf.mo_coeff))


def first_window(system, mol, mf):
    """Pick window orbital indices at the first geometry by irrep (archive Sec. 3.1/3.2)."""
    sym = [mol.irrep_id.index(s) if False else s for s in labels(mol, mf)]
    names = {i: n for i, n in zip(mol.irrep_id, mol.irrep_name)}
    irr = [names[s] for s in sym]
    nocc = mol.nelectron // 2
    occ = list(range(nocc))
    vir = list(range(nocc, len(irr)))

    def lowest(lst, name, k):
        out = [i for i in lst if irr[i] == name][:k]
        assert len(out) == k, (name, k)
        return out

    if system == "H2O":
        core = [0]
        inact_occ = occ[1:]  # 2a1, 1b2, 3a1, 1b1 (all retained)
        # in-plane b irrep = irrep of the lower-energy occupied B orbital
        bocc = [i for i in occ if irr[i] != "A1"]
        b_in = irr[bocc[0]]
        virt = lowest(vir, "A1", 2) + lowest(vir, b_in, 2)  # 4a1,5a1 ; 2b2,3b2
    else:  # N2, D2h
        core = [0, 1]
        inact_occ = occ[2:]  # 2sg, 2su, 1pu x2, 3sg
        virt = (lowest(vir, "B2g", 1) + lowest(vir, "B3g", 1) + lowest(vir, "B1u", 1)
                + lowest(vir, "Ag", 1) + lowest(vir, "B2u", 1) + lowest(vir, "B3u", 1))
    return core, sorted(inact_occ), sorted(virt), irr


def trace(prev, mol, mf, sel):
    """Map previous-geometry orbitals to current ones: same irrep, same occ/vir class, max overlap."""
    pmol, pmf = prev
    S = gto.intor_cross("int1e_ovlp", pmol, mol)
    O = np.abs(pmf.mo_coeff.T @ S @ mf.mo_coeff)
    pirr = [dict(zip(pmol.irrep_id, pmol.irrep_name))[s] for s in labels(pmol, pmf)]
    irr = [dict(zip(mol.irrep_id, mol.irrep_name))[s] for s in labels(mol, mf)]
    nocc = mol.nelectron // 2
    out, used = [], set()
    for p in sel:
        cls = range(nocc) if p < nocc else range(nocc, len(irr))
        cand = [i for i in cls if irr[i] == pirr[p] and i not in used]
        best = max(cand, key=lambda i: O[p, i])
        out.append(best); used.add(best)
    return out, irr


def by_counts(mol, mf, ncore, counts, prev=None, prev_sel=None):
    """Keep the core frozen, retain every other occupied orbital, and fill each irrep up to its
    equilibrium count with the lowest virtuals of that irrep (keeps degenerate pairs together)."""
    irr = [dict(zip(mol.irrep_id, mol.irrep_name))[s] for s in labels(mol, mf)]
    nocc = mol.nelectron // 2
    core = list(range(ncore))
    occ = list(range(ncore, nocc))
    left = dict(counts)
    for i in occ:
        left[irr[i]] -= 1
    vir = []
    if prev is not None:  # projection of each virtual onto the previous retained space
        pmol, pmf = prev
        S = gto.intor_cross("int1e_ovlp", pmol, mol)
        O = pmf.mo_coeff[:, prev_sel].T @ S @ mf.mo_coeff
        score = (O ** 2).sum(0)
    for name, k in left.items():
        assert k >= 0, (name, k)
        cand = [i for i in range(nocc, len(irr)) if irr[i] == name]
        if prev is not None:
            cand = sorted(cand, key=lambda i: -score[i])
        vir += cand[:k]
    return core + occ + sorted(vir), irr


def window_hamiltonian(mol, mf, core, win):
    ncas = len(win)
    nelec = mol.nelectron - 2 * len(core)
    mc = mcscf.CASCI(mf, ncas, nelec)
    order = list(core) + list(win) + [i for i in range(mf.mo_coeff.shape[1]) if i not in core and i not in win]
    mo = mf.mo_coeff[:, order]
    h1, ecore = mc.get_h1eff(mo)
    eri = ao2mo.restore(1, mc.get_h2eff(mo), ncas)
    e_fci, civec = fci.direct_spin1.kernel(h1, eri, ncas, nelec, ecore=ecore, conv_tol=1e-12, max_cycle=300, nroots=1)
    return dict(h1=h1, eri=eri, ecore=ecore, ncas=ncas, nelec=nelec, e_fci=e_fci, civec=civec)


SYSTEMS = {
    "H2O": dict(geoms=[0.9584, 1.75, 3.0], atom=water, group=True, guess="minao", newton=False,
                ref=[(-76.026741428, -76.059281484), (-75.669624909, -75.826835961), (-75.430561431, -75.732177025)]),
    "N2": dict(geoms=[1.10, 1.80, 3.00], atom=n2, group="D2h", guess="1e", newton=True,
               ref=[(-108.953796253, -109.059444517), (-108.341640217, -108.806606966), (-108.231992299, -108.749118387)]),
}


def build_all(system):
    spec = SYSTEMS[system]
    out, prev, sel = [], None, None
    for k, r in enumerate(spec["geoms"]):
        mol, mf = run_rhf(spec["atom"](r), spec["group"], spec["guess"], spec["newton"])
        if prev is None:
            core, occ, vir, irr = first_window(system, mol, mf)
            sel = core + occ + vir
            new = sel
            counts = {}
            for i in occ + vir:
                counts[irr[i]] = counts.get(irr[i], 0) + 1
        else:
            new, irr = by_counts(mol, mf, len(core), counts, prev, sel[len(core):])
        ncore = 1 if system == "H2O" else 2
        core, win = new[:ncore], new[ncore:]
        nocc = mol.nelectron // 2
        win = sorted([i for i in win if i < nocc]) + sorted([i for i in win if i >= nocc])
        W = window_hamiltonian(mol, mf, core, win)
        W.update(system=system, R=r, e_hf=mf.e_tot, mol=mol, mf=mf, core=core, win=win,
                 win_irreps=[irr[i] for i in win], mo_energy=mf.mo_energy[win])
        W["ref_hf"], W["ref_fci"] = spec["ref"][k]
        out.append(W)
        prev, sel = (mol, mf), new
    return out


if __name__ == "__main__":
    import sys
    for s in sys.argv[1:] or ["H2O", "N2"]:
        for W in build_all(s):
            print(f"{s} R={W['R']:.4f} win={W['win_irreps']} "
                  f"dHF={1e3*(W['e_hf']-W['ref_hf']):+.4f} mHa dFCI={1e3*(W['e_fci']-W['ref_fci']):+.4f} mHa", flush=True)


Overwriting windows.py


### `paulis.py` — Pauli algebra, Hamiltonian, pool, gradient operators

In [2]:
%%writefile paulis.py
"""Pauli algebra in (x, z) bit-mask form, JW Hamiltonian, UCCSD pool and gradient operators.

Conventions follow the Part I notebook: qubit q is bit (nq-1-q); spin orbital 2p = alpha, 2p+1 = beta;
P(x, z) = i^{|x&z|} X^x Z^z, so Y = iXZ.
"""
import itertools
import numpy as np
import scipy.sparse as sp
import openfermion as of
from openfermion.chem.molecular_data import spinorb_from_spatial

U = np.uint64


def pc(a):
    return np.bitwise_count(np.asarray(a, dtype=U)).astype(np.int64)


def term_to_xz(term, nq):
    x = z = 0
    for q, p in term:
        bit = 1 << (nq - 1 - q)
        if p in "XY":
            x |= bit
        if p in "ZY":
            z |= bit
    return x, z


def qop_to_arrays(op, nq, tol=1e-12):
    xs, zs, cs = [], [], []
    for t, c in op.terms.items():
        if abs(c) < tol or len(t) == 0 and False:
            continue
        x, z = term_to_xz(t, nq)
        xs.append(x); zs.append(z); cs.append(c)
    return np.array(xs, dtype=U), np.array(zs, dtype=U), np.array(cs, dtype=complex)


def hamiltonian(W):
    h1, eri, ecore = W["h1"], W["eri"], W["ecore"]
    two = np.asarray(eri.transpose(0, 2, 3, 1), order="C")
    one_so, two_so = spinorb_from_spatial(h1, two)
    iop = of.InteractionOperator(ecore, one_so, 0.5 * two_so)
    H = of.jordan_wigner(of.get_fermion_operator(iop))
    H.compress(1e-10)
    nq = 2 * h1.shape[0]
    x, z, c = qop_to_arrays(H, nq)
    return nq, x, z, c.real


def uccsd_pool(nq, nelec):
    occ = list(range(nelec)); vir = list(range(nelec, nq))
    pool, labels = [], []
    for i in occ:
        for a in vir:
            if i % 2 == a % 2:
                op = of.FermionOperator(((a, 1), (i, 0))) - of.FermionOperator(((i, 1), (a, 0)))
                pool.append(op); labels.append(f"S {i}->{a}")
    for i, j in itertools.combinations(occ, 2):
        for a, b in itertools.combinations(vir, 2):
            if (i % 2 + j % 2) == (a % 2 + b % 2):
                if {i % 2, j % 2} != {a % 2, b % 2} and (i % 2 + j % 2) == 1:
                    continue
                op = of.FermionOperator(((b, 1), (a, 1), (j, 0), (i, 0)))
                op = op - of.hermitian_conjugated(op)
                pool.append(op); labels.append(f"D {i},{j}->{a},{b}")
    out = []
    for op in pool:
        q = of.jordan_wigner(op); q.compress(1e-12)
        out.append(qop_to_arrays(q, nq))
    return out, labels


def mult(x1, z1, x2, z2):
    """P1*P2 = i^k P3 with k returned mod 4 (elementwise, broadcast)."""
    x3, z3 = x1 ^ x2, z1 ^ z2
    k = pc(x1 & z1) + pc(x2 & z2) - pc(x3 & z3) + 2 * pc(z1 & x2)
    return x3, z3, np.mod(k, 4)


IK = np.array([1, 1j, -1, -1j])


def commutator(Hx, Hz, Hc, Gx, Gz, Gc):
    """[H, G] for H Hermitian (real coeffs) and G anti-Hermitian (imag coeffs). Returns real coeffs."""
    X, Z, C = [], [], []
    for gx, gz, gc in zip(Gx, Gz, Gc):
        anti = (pc((Hx & gz) ^ (Hz & gx)) & 1).astype(bool)
        hx, hz, hc = Hx[anti], Hz[anti], Hc[anti]
        x3, z3, k = mult(hx, hz, gx, gz)
        X.append(x3); Z.append(z3); C.append(2 * hc * gc * IK[k])
    X = np.concatenate(X); Z = np.concatenate(Z); C = np.concatenate(C)
    key = (X << U(32)) | Z
    uk, inv = np.unique(key, return_inverse=True)
    cr = np.bincount(inv, weights=C.real, minlength=len(uk))
    ci = np.bincount(inv, weights=C.imag, minlength=len(uk))
    assert np.abs(ci).max(initial=0) < 1e-9
    keep = np.abs(cr) > 1e-10
    return uk[keep], cr[keep]


def gradient_matrix(nq, Hx, Hz, Hc, pool):
    keys, coefs = [], []
    for gx, gz, gc in pool:
        k, c = commutator(Hx, Hz, Hc, gx, gz, gc)
        keys.append(k); coefs.append(c)
    allk = np.unique(np.concatenate(keys))
    rows, cols, vals = [], [], []
    for i, (k, c) in enumerate(zip(keys, coefs)):
        rows.append(np.full(len(k), i)); cols.append(np.searchsorted(allk, k)); vals.append(c)
    A = sp.csr_matrix((np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))), shape=(len(pool), len(allk)))
    xs = allk >> U(32); zs = allk & U(0xFFFFFFFF)
    return A, xs.astype(U), zs.astype(U)


def label(x, z, nq):
    s = []
    for q in range(nq):
        b = 1 << (nq - 1 - q)
        s.append({(0, 0): "I", (1, 0): "X", (0, 1): "Z", (1, 1): "Y"}[(bool(int(x) & b), bool(int(z) & b))])
    return "".join(s)


Overwriting paulis.py


### `states.py` — HF, CISD and FCI states as sparse vectors

In [3]:
%%writefile states.py
"""HF, CISD and FCI states of a window, as sparse JW bit-string vectors."""
import numpy as np
import scipy.sparse.linalg as sla
from pyscf import fci
from pyscf.fci import cistring
from meas import State
from paulis import U, pc


def to_jw(civec, norb, nelec, tol=1e-14):
    """pyscf (alpha-string, beta-string) CI vector -> JW basis states.
    pyscf orders a+_alpha... a+_beta...; JW/OpenFermion orders by spin-orbital index
    (2k = alpha, 2k+1 = beta). Reordering sign = (-1)^{#(beta j, alpha k) with j < k}."""
    na = nb = nelec // 2
    sa = cistring.make_strings(range(norb), na)
    sb = cistring.make_strings(range(norb), nb)
    nq = 2 * norb
    ia, ib = np.nonzero(np.abs(civec) > tol)
    idx = np.zeros(len(ia), dtype=U); sign = np.ones(len(ia))
    for n, (i, j) in enumerate(zip(ia, ib)):
        a, b = int(sa[i]), int(sb[j])
        bits = 0
        for k in range(norb):
            if a >> k & 1:
                bits |= 1 << (nq - 1 - 2 * k)
            if b >> k & 1:
                bits |= 1 << (nq - 1 - (2 * k + 1))
        cross = 0
        for k in range(norb):
            if a >> k & 1:
                cross += bin(b & ((1 << k) - 1)).count("1")
        idx[n] = bits; sign[n] = -1 if cross & 1 else 1
    return State(idx, civec[ia, ib] * sign)


def hf_state(nq, nelec):
    return State(np.array([sum(1 << (nq - 1 - q) for q in range(nelec))], dtype=U), np.array([1.0]))


def cisd_vector(W, spin_penalty=1.0):
    """Lowest singlet in the space of the HF determinant plus all single and double excitations."""
    norb, nelec = W["ncas"], W["nelec"]
    na = nelec // 2
    sa = cistring.make_strings(range(norb), na)
    hf = (1 << na) - 1
    exc = np.array([bin(int(s) & ~hf).count("1") for s in sa])
    mask = (exc[:, None] + exc[None, :]) <= 2
    sel = np.flatnonzero(mask.ravel())
    shape = (len(sa), len(sa))
    h2e = fci.direct_spin1.absorb_h1e(W["h1"], W["eri"], norb, nelec, 0.5)

    def mv(v):
        full = np.zeros(shape[0] * shape[1]); full[sel] = v
        full = full.reshape(shape)
        out = fci.direct_spin1.contract_2e(h2e, full, norb, nelec)
        out = out + spin_penalty * fci.spin_op.contract_ss(full, norb, nelec)
        return out.ravel()[sel]

    op = sla.LinearOperator((len(sel), len(sel)), matvec=mv, dtype=float)
    e, v = sla.eigsh(op, k=1, which="SA", tol=1e-12)
    full = np.zeros(shape[0] * shape[1]); full[sel] = v[:, 0]
    full = full.reshape(shape)
    e_cisd = fci.direct_spin1.energy(W["h1"], W["eri"], full, norb, nelec) + W["ecore"]
    return full, e_cisd, len(sel)


def energy(state, Hx, Hz, Hc):
    from meas import pauli_expect
    return float(Hc @ pauli_expect(state, Hx, Hz))


Overwriting states.py


### `meas.py` — Grouping, fragment variances, Part I methods

In [4]:
%%writefile meas.py
"""States, fast FC grouping, fragment variances and the Part I oracle methods (M1, M2, M3, M3-R).

The method code (allocate, m1/m2/m3/m3r costs) is copied from the Part I notebook; only the
data structures changed (sparse A, sparse states) so that 16- and 22-qubit windows fit in memory.
"""
import numpy as np
import scipy.sparse as sp
from scipy.stats import norm
from paulis import U, pc

# ---------------------------------------------------------------- grouping

def group_fc(xs, zs, order, nq):
    """First-fit sorted insertion into fully commuting groups (same result as the Part I
    routine). A Pauli commutes with every member of a group iff it commutes with a GF(2) basis
    of the group, so only the basis (<= nq vectors per group) is stored and checked."""
    L = len(xs)
    gid = -np.ones(L, dtype=np.int64)
    cap = 1 << 16
    bx = np.zeros(cap, dtype=U); bz = np.zeros(cap, dtype=U); bg = np.zeros(cap, dtype=np.int64)
    nb = 0
    piv = []  # per group: dict pivot -> (vector)
    ng = 0
    for l in order:
        x, z = xs[l], zs[l]
        if nb:
            bad = (pc((bx[:nb] & z) ^ (bz[:nb] & x)) & 1).astype(bool)
            badg = np.bincount(bg[:nb][bad], minlength=ng)
            ok = np.flatnonzero(badg == 0)
            g = int(ok[0]) if len(ok) else ng
        else:
            g = 0
        if g == ng:
            ng += 1; piv.append({})
        gid[l] = g
        v = (int(x) << 32) | int(z)
        rows = piv[g]
        for p, r in rows.items():
            if v >> p & 1:
                v ^= r
        if v:
            p = v.bit_length() - 1
            for q in list(rows):
                if rows[q] >> p & 1:
                    rows[q] ^= v
            rows[p] = v
            if nb == cap:
                cap *= 2
                bx = np.resize(bx, cap); bz = np.resize(bz, cap); bg = np.resize(bg, cap)
            bx[nb] = U(v >> 32); bz[nb] = U(v & 0xFFFFFFFF); bg[nb] = g; nb += 1
    return gid, ng


def base_partition(A, xs, zs, nq):
    w = np.sqrt(np.asarray(A.multiply(A).sum(0)).ravel())
    gid, ng = group_fc(xs, zs, np.argsort(-w, kind="stable"), nq)
    return split(gid, ng)


def split(gid, ng):
    order = np.argsort(gid, kind="stable")
    cuts = np.searchsorted(gid[order], np.arange(ng + 1))
    return [order[cuts[k]:cuts[k + 1]] for k in range(ng)]


# ---------------------------------------------------------------- sparse states

class State:
    """psi = sum_b amp[b] |b>, with basis states stored as sorted uint64 bit strings."""

    def __init__(self, idx, amp):
        o = np.argsort(idx)
        self.idx = np.asarray(idx, dtype=U)[o]
        self.amp = np.asarray(amp, dtype=complex)[o]
        self.amp /= np.linalg.norm(self.amp)

    def lookup(self, keys):
        pos = np.searchsorted(self.idx, keys)
        pos = np.minimum(pos, len(self.idx) - 1)
        hit = self.idx[pos] == keys
        out = np.zeros(len(keys), dtype=complex)
        out[hit] = self.amp[pos[hit]]
        return out


def pauli_expect(state, xs, zs):
    """<psi|P_l|psi> for many Paulis."""
    out = np.zeros(len(xs))
    ny = pc(xs & zs)
    for b, a in zip(state.idx, state.amp):
        sgn = 1 - 2 * (pc(b & zs) & 1)
        out += (np.conj(state.lookup(b ^ xs)) * a * sgn * (1j) ** ny).real
    return out


def fragment_variances(state, A, xs, zs, groups, batch_pairs=1_000_000):
    """q[i, a] = Var_psi(F_ia), mu[i, a] = <F_ia>, F_ia = sum_{l in group a} A[i, l] P_l.

    F_ia psi = sum_l A_il sum_b psi_b phase(l, b) |b ^ x_l>. Every (Pauli l, det b) pair is sent to
    the bucket (group a, output state b ^ x_l); a sparse product with A then gives all amplitudes."""
    A = sp.csc_matrix(A)
    K, G = A.shape[0], len(groups)
    q = np.zeros((K, G)); mu = np.zeros((K, G))
    nd = len(state.idx)
    per = max(1, batch_pairs // nd)
    gi = 0
    while gi < G:
        # take groups until the pair budget is used
        sel, npau = [], 0
        while gi < G and (npau == 0 or npau + len(groups[gi]) <= per):
            sel.append(gi); npau += len(groups[gi]); gi += 1
        mem = np.concatenate([groups[g] for g in sel])
        gof = np.concatenate([np.full(len(groups[g]), k) for k, g in enumerate(sel)])
        x = xs[mem]; z = zs[mem]; ny = pc(x & z)
        out = (x[:, None] ^ state.idx[None, :])                      # (npau, nd)
        sgn = 1 - 2 * (pc(state.idx[None, :] & z[:, None]) & 1)
        val = sgn * (1j) ** ny[:, None] * state.amp[None, :]
        key = (gof[:, None].astype(U) << U(40)) ^ out                 # groups < 2^23, states < 2^40
        uk, inv = np.unique(key.ravel(), return_inverse=True)
        Phi = sp.csr_matrix((val.ravel(), (np.repeat(np.arange(len(mem)), nd), inv)), shape=(len(mem), len(uk)))
        M = (A[:, mem] @ Phi).tocsr()                                 # K x buckets, amplitudes of F psi
        bgrp = (uk >> U(40)).astype(np.int64)
        bout = uk & U((1 << 40) - 1)
        cpsi = np.conj(state.lookup(bout))
        Mc = M.tocoo()
        norm2 = np.zeros((K, len(sel))); mean = np.zeros((K, len(sel)))
        np.add.at(norm2, (Mc.row, bgrp[Mc.col]), np.abs(Mc.data) ** 2)
        np.add.at(mean, (Mc.row, bgrp[Mc.col]), (cpsi[Mc.col] * Mc.data).real)
        q[:, sel] = np.maximum(norm2 - mean ** 2, 0.0)
        mu[:, sel] = mean
    return q, mu


# ---------------------------------------------------------------- Part I methods (copied)

def z_value(n_arms, delta=0.05):
    return norm.ppf(1 - delta / (2 * n_arms))


def allocate(q, eps, iters=400, tol=1e-9):
    q = np.asarray(q, float)
    need = q.sum(0) > 0
    q = q[:, need]
    eps2 = np.asarray(eps, float) ** 2
    lam = np.ones(q.shape[0])
    best = None
    for it in range(iters):
        n = np.sqrt(np.maximum(lam @ q, 1e-300))
        V = (q / n).sum(1)
        scale = np.max(V / eps2)
        cost = scale * n.sum()
        if best is None or cost < best[0] - tol * best[0]:
            best = (cost, n * scale)
        lam = lam * np.sqrt(V / eps2)
        lam /= lam.max()
    out = np.zeros(len(need))
    out[need] = best[1]
    return best[0], out


def gaps(g, delta=0.0):
    """delta > 0 gives the Delta-good target: arms within delta of the best need not be separated,
    so every elimination radius is at least delta/2."""
    a = np.abs(g)
    w = int(np.argmax(a))
    s = np.sort(a)[::-1]
    gap = max(s[0] - s[1], delta)
    D = np.maximum(a[w] - a, delta)
    D[w] = gap
    return w, gap, D


def m1_cost(q, g, z, delta=0.0):
    w, gap, D = gaps(g, delta)
    eps = np.full(q.shape[0], gap / 2 / z)
    active = q.sum(1) > 0
    return allocate(q[active], eps[active])


def m2_cost(q_list, g, z, delta=0.0):
    w, gap, D = gaps(g, delta)
    per = np.zeros(len(g))
    for i, qi in enumerate(q_list):
        s = np.sqrt(qi).sum()
        if s == 0:
            continue
        per[i] = s ** 2 / (D[i] / 2 / z) ** 2
    return per.sum(), per


def m3_cost(q, g, z, delta=0.0):
    w, gap, D = gaps(g, delta)
    levels = np.unique(np.round(D, 12))[::-1]
    charged = np.zeros(q.shape[1])
    stages = []
    for d in levels:
        act = np.flatnonzero((D <= d + 1e-12) & (q.sum(1) > 0))
        if len(act) == 0:
            continue
        cost, n = allocate(q[act], np.full(len(act), d / 2 / z))
        charged = np.maximum(charged, n)
        stages.append(dict(radius=d / 2, active=len(act), stage_cost=cost))
    return charged.sum(), charged, stages


def per_gradient_groups(A, xs, zs, nq):
    A = sp.csr_matrix(A)
    out = []
    for i in range(A.shape[0]):
        sup = A.indices[A.indptr[i]:A.indptr[i + 1]]
        c = A.data[A.indptr[i]:A.indptr[i + 1]]
        o = np.argsort(sup); sup, c = sup[o], c[o]
        gid, ng = group_fc(xs[sup], zs[sup], np.argsort(-np.abs(c), kind="stable"), nq)
        out.append([sup[m] for m in split(gid, ng)])
    return out


def shared_fraction(A, active):
    cnt = np.asarray((abs(A[active]) > 0).sum(0)).ravel()
    union = int((cnt > 0).sum())
    return (cnt >= 2).sum() / max(union, 1), union


def regroup(A, xs, zs, active, nq):
    sub = A[active]
    sup = np.flatnonzero(np.asarray((abs(sub) > 0).sum(0)).ravel() > 0)
    w = np.sqrt(np.asarray(sub[:, sup].multiply(sub[:, sup]).sum(0)).ravel())
    gid, ng = group_fc(xs[sup], zs[sup], np.argsort(-w, kind="stable"), nq)
    return [sup[m] for m in split(gid, ng)]


def m3r_cost(state, xs, zs, A, q, g, z, nq, threshold=0.0, delta=0.0):
    A = sp.csr_matrix(A)
    w, gap, D = gaps(g, delta)
    levels = np.unique(np.round(D, 12))[::-1]
    parent = np.zeros(q.shape[1])
    fresh, qn, groups_new, trigger, v_old, rows = None, None, None, None, None, None
    stages = []
    for d in levels:
        act = np.flatnonzero((D <= d + 1e-12) & (q.sum(1) > 0))
        if len(act) == 0:
            continue
        frac, union = shared_fraction(A, act)
        if groups_new is None and frac <= threshold:
            groups_new = regroup(A, xs, zs, act, nq)
            qn, _ = fragment_variances(state, A[act], xs, zs, groups_new)
            rows = {i: r for r, i in enumerate(act)}
            fresh = np.zeros(len(groups_new))
            v_old = {}
            for i in act:
                used = q[i] > 0
                v_old[i] = np.sum(q[i][used] / parent[used]) if np.all(parent[used] > 0) else np.inf
            trigger = dict(radius=d / 2, active=len(act), union_terms=union, new_groups=len(groups_new))
        eps = np.full(len(act), d / 2 / z)
        if groups_new is None:
            cost, n = allocate(q[act], eps)
            parent = np.maximum(parent, n)
        else:
            need = []
            for i, e in zip(act, eps):
                inv = 1.0 / e ** 2 - (1.0 / v_old[i] if np.isfinite(v_old[i]) else 0.0)
                need.append(1.0 / inv if inv > 0 else np.inf)
            need = np.array(need)
            keep = np.isfinite(need)
            if keep.any():
                cost, n = allocate(qn[[rows[i] for i in act[keep]]], np.sqrt(need[keep]))
                fresh = np.maximum(fresh, n)
            else:
                cost = 0.0
        stages.append(dict(radius=d / 2, active=len(act), shared_fraction=float(frac),
                           regrouped=groups_new is not None, stage_cost=cost))
    total = parent.sum() + (fresh.sum() if fresh is not None else 0.0)
    return total, stages, trigger, groups_new


def fragmentation(A, groups, i):
    A = sp.csr_matrix(A)
    gid = np.empty(A.shape[1], dtype=np.int64)
    for a, mem in enumerate(groups):
        gid[mem] = a
    return len(set(gid[A.indices[A.indptr[i]:A.indptr[i + 1]]]))


Overwriting meas.py


### `circuits.py` — Two-qubit gate counts of the measurement circuits

In [5]:
%%writefile circuits.py
"""Clifford diagonalization gate counts (Part I notebook routine, gate list only)."""
import numpy as np


def gf2_basis(xs, zs, nq):
    rows, piv = [], []
    for x, z in zip(xs, zs):
        v = (int(x) << nq) | int(z)
        for p, r in zip(piv, rows):
            if v >> p & 1:
                v ^= r
        if v:
            p = v.bit_length() - 1
            for k in range(len(rows)):
                if rows[k] >> p & 1:
                    rows[k] ^= v
            rows.append(v); piv.append(p)
    X = np.array([[(r >> (2 * nq - 1 - q)) & 1 for q in range(nq)] for r in rows], dtype=np.uint8).reshape(-1, nq)
    Z = np.array([[(r >> (nq - 1 - q)) & 1 for q in range(nq)] for r in rows], dtype=np.uint8).reshape(-1, nq)
    return X, Z


def two_qubit_count(xs, zs, nq):
    X, Z = gf2_basis(xs, zs, nq)
    k = X.shape[0]
    n2 = 0

    def h(q):
        X[:, q], Z[:, q] = Z[:, q].copy(), X[:, q].copy()

    def rref():
        pivots, r = [], 0
        for q in range(nq):
            rows = np.flatnonzero(X[r:, q]) + r
            if len(rows) == 0:
                continue
            p = rows[0]
            if p != r:
                X[[r, p]] = X[[p, r]]; Z[[r, p]] = Z[[p, r]]
            for rr in np.flatnonzero(X[:, q]):
                if rr != r:
                    X[rr] ^= X[r]; Z[rr] ^= Z[r]
            pivots.append(q); r += 1
            if r == k:
                break
        return pivots

    while True:
        pivots = rref()
        if len(pivots) == k:
            break
        r = len(pivots)
        free = [q for q in range(nq) if Z[r, q] and q not in pivots]
        h(free[0])
    for r, p in enumerate(pivots):
        for q in np.flatnonzero(X[r]):
            if q != p:
                n2 += 1; X[:, q] ^= X[:, p]; Z[:, p] ^= Z[:, q]
    for r, p in enumerate(pivots):
        if Z[r, p]:
            Z[:, p] ^= X[:, p]
    for a in range(k):
        for b in range(a + 1, k):
            if Z[a, pivots[b]]:
                n2 += 1; Z[:, pivots[a]] ^= X[:, pivots[b]]; Z[:, pivots[b]] ^= X[:, pivots[a]]
    return n2


Overwriting circuits.py


### `run_partI.py` — Part I runner

In [6]:
%%writefile run_partI.py
"""Part I (M1, M2, M3, M3-R) on one cc-pVDZ window geometry, HF and CISD states.
Usage: python run_partI.py H2O 0   (geometry index into windows.SYSTEMS)"""
import os, sys, json, time
os.environ.setdefault("OMP_NUM_THREADS", "2")
import numpy as np, scipy.sparse as sp
from windows import build_all, SYSTEMS
from paulis import hamiltonian, uccsd_pool, gradient_matrix, label
from meas import *
from states import hf_state, cisd_vector, to_jw, energy
from circuits import two_qubit_count

system, k = sys.argv[1], int(sys.argv[2])
POOL = sys.argv[3] if len(sys.argv) > 3 else "full"
DELTA = 1e-3
OUT = "results/partI"; os.makedirs(OUT, exist_ok=True)
t0 = time.time()
log = lambda *a: print(f"[{time.time()-t0:7.1f}s]", *a, flush=True)

W = [w for w in build_all(system) if abs(w["R"] - SYSTEMS[system]["geoms"][k]) < 1e-9][0]
tag0 = f"{system}_{W['R']:.4g}_{POOL}"
nq, Hx, Hz, Hc = hamiltonian(W); ne = W["nelec"]
pool, labels = uccsd_pool(nq, ne)
if POOL == "sym":  # keep generators that are totally symmetric (irrep ids combine by XOR in pyscf)
    mol = W["mol"]
    ids = [mol.irrep_id[mol.irrep_name.index(n)] for n in W["win_irreps"]]
    def sym_ok(lab):
        a, b = lab.split(" ", 1)[1].split("->")
        x = 0
        for o in a.split(",") + b.split(","):
            x ^= ids[int(o) // 2]
        return x == 0
    keep = [i for i, l in enumerate(labels) if sym_ok(l)]
    pool = [pool[i] for i in keep]; labels = [labels[i] for i in keep]
A, xs, zs = gradient_matrix(nq, Hx, Hz, Hc, pool); A = A.tocsr()
log(tag0, "nq", nq, "H terms", len(Hx), "pool", len(pool), "L", A.shape[1], "nnz", A.nnz)

groups = base_partition(A, xs, zs, nq); log("parent groups", len(groups))
n2q = np.array([two_qubit_count(xs[m], zs[m], nq) for m in groups]); log("circuits done, mean 2q", n2q.mean())
pg = per_gradient_groups(A, xs, zs, nq); log("per-gradient groups", sum(len(p) for p in pg))

sp.save_npz(f"{OUT}/{tag0}_A.npz", A)
np.savez_compressed(f"{OUT}/{tag0}_paulis.npz", xs=xs, zs=zs, nq=nq)
json.dump(dict(generators=labels, window_irreps=W["win_irreps"], R=W["R"], e_hf=W["e_hf"], e_fci_window=W["e_fci"],
               convention="sorted insertion by decreasing ||A[:,l]||_2 (stable), first-fit FC groups; no randomness",
               groups=[m.tolist() for m in groups], n_2q=n2q.tolist()),
          open(f"{OUT}/{tag0}_M1_FCUG_M3_BAIFCUG_groups.json", "w"))

civ, e_cisd, dim = cisd_vector(W)
states = {"HF": hf_state(nq, ne), "CISD": to_jw(civ, W["ncas"], ne)}
rows = []
for st, psi in states.items():
    e_st = energy(psi, Hx, Hz, Hc)
    mu_all = pauli_expect(psi, xs, zs); g = A @ mu_all
    z = z_value(len(g))
    q, mu = fragment_variances(psi, A, xs, zs, groups)
    err = np.abs(mu.sum(1) - g).max(); assert err < 1e-8, err
    log(st, "E-E_FCI(mHa)", 1e3 * (e_st - W["e_fci"]), "fragment check", err)
    ql = [fragment_variances(psi, A[i], xs, zs, pg[i])[0][0] for i in range(len(g))]
    c1, n1 = m1_cost(q, g, z)
    c2, per2 = m2_cost(ql, g, z)
    c3, n3, stages = m3_cost(q, g, z)
    c3r, stages_r, trig, groups_r = m3r_cost(psi, xs, zs, A, q, g, z, nq)
    c1d, _ = m1_cost(q, g, z, DELTA)
    c2d, _ = m2_cost(ql, g, z, DELTA)
    c3d, _, stages_d = m3_cost(q, g, z, DELTA)
    w, gap, D = gaps(g); order = np.argsort(-np.abs(g))
    top2 = order[:2]
    tag = f"{tag0}_{st}"
    np.savez_compressed(f"{OUT}/{tag}_M1_FCUG.npz", fragment_variances=q, shots_per_context=n1, total=c1, z=z)
    np.savez_compressed(f"{OUT}/{tag}_M2_BAIFCIG.npz", total=c2, per_arm=per2, z=z)
    np.savez_compressed(f"{OUT}/{tag}_M3_BAIFCUG.npz", shots_per_context=n3, total=c3, z=z)
    json.dump(stages, open(f"{OUT}/{tag}_M3_BAIFCUG_stages.json", "w"), default=float)
    json.dump(dict(trigger=trig, stages=stages_r), open(f"{OUT}/{tag}_M3R_BAIFCUG_regroup.json", "w"), default=float)
    json.dump(dict(gradients=g.tolist(), labels=labels), open(f"{OUT}/{tag}_exact_gradients.json", "w"))
    row = dict(system=system, R=W["R"], state=st, qubits=nq, electrons=ne, H_terms=len(Hx), pool=len(pool),
               universal_terms=A.shape[1], contexts=len(groups), M2_groups=int(sum(len(p) for p in pg)),
               mean_2q=float(n2q.mean()), E_state_minus_FCI_mHa=1e3 * (e_st - W["e_fci"]),
               nonzero_g=int(np.sum(np.abs(g) > 1e-8)), top=labels[order[0]], second=labels[order[1]],
               g1=float(abs(g[order[0]])), g2=float(abs(g[order[1]])), gap=float(gap), z=float(z),
               M1=c1, M2=c2, M3=c3, M3R=c3r, M1_Delta=c1d, M2_Delta=c2d, M3_Delta=c3d,
               n_Delta_good=int(np.sum(np.abs(g) >= np.abs(g).max() - DELTA)),
               regroup_active=trig["active"] if trig else 0, regroup_contexts=trig["new_groups"] if trig else 0,
               winner_parent_contexts=fragmentation(A, groups, top2[0]), winner_own_groups=len(pg[top2[0]]),
               runner_parent_contexts=fragmentation(A, groups, top2[1]), runner_own_groups=len(pg[top2[1]]),
               final_stage_share_M3=float(stages[-1]["stage_cost"] / c3), seconds=time.time() - t0)
    row["pool_type"] = POOL
    rows.append(row)
    log(st, f"top {row['top']} g1={row['g1']:.6f} g2={row['g2']:.6f} M1={c1:.4g} M2={c2:.4g} M3={c3:.4g} M3R={c3r:.4g} | Delta: M1={c1d:.4g} M2={c2d:.4g} M3={c3d:.4g}")
json.dump(rows, open(f"{OUT}/{tag0}_summary.json", "w"), indent=1, default=float)
log("done")


Overwriting run_partI.py


### `run_contrast.py` — M3-C and Δ-good M3-R runner

In [7]:
%%writefile run_contrast.py
"""Oracle contrast version of M3 on the Part I parent groups (called M3-C here).

M3 eliminates arm j when the two individual confidence intervals of the leader w and of j
separate: each gradient needs Var <= (Delta_j / 2z)^2. The Part II contrast rule instead puts one
interval on the contrast c_j = s_w g_w - s_j g_j = |g_w| - |g_j| and eliminates j when it is
positive: Var(c_j) <= (Delta_j / z)^2. Because w and j are read from the same parent groups,
Var(c_j) includes their covariance. No coefficient splitting and no finite-shot noise: this is
the oracle limit of the II-E target on the unchanged Part I measurement contexts.

Usage: python run_contrast.py H2O 0 full
"""
import os, sys, json, time
import numpy as np, scipy.sparse as sp
from windows import build_all, SYSTEMS
from paulis import hamiltonian, uccsd_pool
from meas import fragment_variances, pauli_expect, z_value, allocate, gaps, m3r_cost
from states import hf_state, cisd_vector, to_jw

system, k, POOL = sys.argv[1], int(sys.argv[2]), sys.argv[3]
DELTA = 1e-3
IN = "results/partI"
W = [w for w in build_all(system) if abs(w["R"] - SYSTEMS[system]["geoms"][k]) < 1e-9][0]
tag0 = f"{system}_{W['R']:.4g}_{POOL}"
A = sp.load_npz(f"{IN}/{tag0}_A.npz").tocsr()
P = np.load(f"{IN}/{tag0}_paulis.npz"); xs, zs, nq = P["xs"], P["zs"], int(P["nq"])
groups = [np.array(m, dtype=np.int64) for m in json.load(open(f"{IN}/{tag0}_M1_FCUG_M3_BAIFCUG_groups.json"))["groups"]]
ne = W["nelec"]
civ, _, _ = cisd_vector(W)
states = {"HF": hf_state(nq, ne), "CISD": to_jw(civ, W["ncas"], ne)}


def m3c_cost(qc, D, w, z):
    levels = np.unique(np.round(D, 12))[::-1]
    charged = np.zeros(qc.shape[1]); stages = []
    for d in levels:
        act = np.flatnonzero((D <= d + 1e-12) & (np.arange(len(D)) != w) & (qc.sum(1) > 0))
        if len(act) == 0:
            continue
        cost, n = allocate(qc[act], np.full(len(act), d / z))
        charged = np.maximum(charged, n)
        stages.append(dict(radius=d, active=len(act) + 1, stage_cost=cost))
    return charged.sum(), stages


out = []
for st, psi in states.items():
    g = A @ pauli_expect(psi, xs, zs)
    z = z_value(len(g))
    s = np.sign(g); s[s == 0] = 1
    w = int(np.argmax(np.abs(g)))
    C = (sp.csr_matrix(np.full((A.shape[0], 1), s[w])) @ A[w] - sp.diags(s) @ A).tocsr()  # row j: s_w A_w - s_j A_j
    qc, _ = fragment_variances(psi, C, xs, zs, groups)
    row = dict(system=system, R=W["R"], state=st, pool_type=POOL)
    for name, dl in (("", 0.0), ("_Delta", DELTA)):
        _, gap, D = gaps(g, dl)
        c, stages = m3c_cost(qc, D, w, z)
        row["M3C" + name] = c
        json.dump(stages, open(f"{IN}/{tag0}_{st}_M3C{name}_stages.json", "w"), default=float)
    q = np.load(f"{IN}/{tag0}_{st}_M1_FCUG.npz")["fragment_variances"]
    row["M3R_Delta"] = m3r_cost(psi, xs, zs, A, q, g, z, nq, delta=DELTA)[0]
    out.append(row)
    print(row, flush=True)
json.dump(out, open(f"{IN}/{tag0}_contrast.json", "w"), indent=1, default=float)


Overwriting run_contrast.py


### `tables.py` — Result tables

In [8]:
%%writefile tables.py
"""Collect the saved Part I results into tables."""
import json, glob
import numpy as np
import pandas as pd

IN = "results/partI"
ORDER = [("H2O", 0.9584), ("H2O", 1.75), ("N2", 1.1), ("N2", 1.8), ("N2", 3.0)]


def load(pool_type=None):
    rows = []
    for f in sorted(glob.glob(f"{IN}/*_summary.json")):
        rows += json.load(open(f))
    df = pd.DataFrame(rows)
    con = []
    for f in sorted(glob.glob(f"{IN}/*_contrast.json")):
        con += json.load(open(f))
    if con:
        df = df.merge(pd.DataFrame(con), on=["system", "R", "state", "pool_type"], how="left")
    df["key"] = [ORDER.index((s, round(r, 4))) for s, r in zip(df.system, df.R)]
    df = df.sort_values(["pool_type", "key", "state"], ascending=[False, True, False]).reset_index(drop=True)
    df["case"] = [f"{s} {r:g} Å" for s, r in zip(df.system, df.R)]
    if pool_type:
        df = df[df.pool_type == pool_type].reset_index(drop=True)
    return df


def sci(x):
    return f"{x:.2e}" if np.isfinite(x) else "–"


def structure(df):
    d = df.drop_duplicates(["system", "R", "pool_type"])
    return pd.DataFrame(dict(Case=d.case, Pool=d.pool_type, Qubits=d.qubits, Electrons=d.electrons, H_terms=d.H_terms,
                             Generators=d.pool, Pauli_terms=d.universal_terms, FC_contexts=d.contexts,
                             M2_groups=d.M2_groups, mean_2q=d.mean_2q.round(1))).reset_index(drop=True)


def gradients(df):
    return pd.DataFrame(dict(Case=df.case, Pool=df.pool_type, State=df.state,
                             E_minus_FCI_mHa=df.E_state_minus_FCI_mHa.round(2), Top=df.top, Second=df.second,
                             g1=df.g1.round(6), g2=df.g2.round(6), gap=df.gap.map(lambda x: f"{x:.2e}"),
                             nonzero_g=df.nonzero_g, n_Delta_good=df.n_Delta_good))


def methods(df, delta=False):
    s = "_Delta" if delta else ""
    out = pd.DataFrame(dict(Case=df.case, Pool=df.pool_type, State=df.state,
                            M1=df["M1" + s].map(sci), M2=df["M2" + s].map(sci), M3=df["M3" + s].map(sci)))
    if not delta:
        out["M3R"] = df.M3R.map(sci)
    elif "M3R_Delta" in df:
        out["M3R"] = df.M3R_Delta.map(sci)
    if "M3C" + s in df:
        out["M3C"] = df["M3C" + s].map(sci)
    out["M1/M2"] = (df["M1" + s] / df["M2" + s]).round(2)
    out["M2/M3"] = (df["M2" + s] / df["M3" + s]).round(2)
    r3 = df.M3R if not delta else df.get("M3R_Delta")
    if r3 is not None:
        out["M2/M3R"] = (df["M2" + s] / r3).round(2)
    if "M3C" + s in df:
        out["M3/M3C"] = (df["M3" + s] / df["M3C" + s]).round(2)
    return out


Overwriting tables.py


## 1. Window validation against the archive table

In [9]:
import os
os.environ["OMP_NUM_THREADS"] = "2"
import pandas as pd
from windows import build_all
rows = []
for s in ("H2O", "N2"):
    for W in build_all(s):
        if s == "H2O" and W["R"] == 3.0:
            continue
        rows.append(dict(System=s, R=W["R"], window=" ".join(W["win_irreps"]),
                         dE_RHF_uHa=round(1e6 * (W["e_hf"] - W["ref_hf"]), 2),
                         dE_FCI_uHa=round(1e6 * (W["e_fci"] - W["ref_fci"]), 2)))
pd.DataFrame(rows)

,System,R,window,dE_RHF_uHa,dE_FCI_uHa
0,H2O,0.9584,A1 B2 A1 B1 A1 B2 B2 A1,0.00,-0.00
1,H2O,1.7500,A1 B1 B2 A1 A1 B2 A1 B2,0.05,0.04
2,N2,1.1000,Ag B1u Ag B3u B2u B2g B3g B1u Ag B3u B2u,0.01,-0.54
3,N2,1.8000,Ag B1u B3u B2g B2u Ag B3g B1u B3u B2u Ag,0.01,0.03
4,N2,3.0000,Ag B1u B2u B3g B2g B3u Ag B1u Ag B2u B3u,-0.00,0.06


## 2. Runs (switched off; each geometry takes 1–10 min on 2 cores)

In [10]:
RUN = False
import subprocess
jobs = [("H2O", 0, "full"), ("H2O", 1, "full"), ("H2O", 0, "sym"), ("H2O", 1, "sym"),
        ("N2", 0, "sym"), ("N2", 1, "sym"), ("N2", 2, "sym")]
if RUN:
    for s, k, p in jobs:
        subprocess.run(["python3", "run_partI.py", s, str(k), p], check=True)
        subprocess.run(["python3", "run_contrast.py", s, str(k), p], check=True)

In [11]:
import pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
from tables import load, structure, gradients, methods
df = load()

## 3. Observable structure

In [12]:
structure(df)

,Case,Pool,Qubits,Electrons,H_terms,Generators,Pauli_terms,FC_contexts,M2_groups,mean_2q
0,H2O 0.9584 Å,sym,16,8,1905,112,61303,1467,11717,23.8
1,H2O 1.75 Å,sym,16,8,1905,112,61303,1449,11842,24.3
2,N2 1.1 Å,sym,22,10,2850,172,206547,4282,19179,45.6
3,N2 1.8 Å,sym,22,10,2850,170,207473,3304,18910,31.2
4,N2 3 Å,sym,22,10,2850,166,202221,3979,18718,42.1
5,H2O 0.9584 Å,full,16,8,1905,360,220355,4945,37980,29.8
6,H2O 1.75 Å,full,16,8,1905,360,220355,4911,38712,29.9


## 4. Leading gradients and how hard the selection is
`n_Delta_good` = number of generators within 1 mHa of the best.

In [13]:
gradients(df)

,Case,Pool,State,E_minus_FCI_mHa,Top,Second,g1,g2,gap,nonzero_g,n_Delta_good
0,H2O 0.9584 Å,sym,HF,32.54,"D 2,3->12,13","D 2,3->14,15",0.179271,0.146089,3.32e-02,100,1
1,H2O 0.9584 Å,sym,CISD,0.40,"D 2,3->12,13",S 2->10,0.002195,0.002189,6.21e-06,112,11
2,H2O 1.75 Å,sym,HF,157.21,"D 4,5->10,11","D 6,7->8,9",0.274445,0.245186,2.93e-02,108,1
3,H2O 1.75 Å,sym,CISD,16.49,"D 4,5->10,11","D 6,7->8,9",0.052233,0.043816,8.42e-03,112,1
4,N2 1.1 Å,sym,HF,105.65,"D 6,7->10,11","D 8,9->12,13",0.274847,0.274847,1.42e-08,172,2
5,N2 1.1 Å,sym,CISD,5.53,"D 6,7->10,11","D 8,9->12,13",0.020173,0.020173,1.44e-09,172,2
6,N2 1.8 Å,sym,HF,464.97,"D 8,9->12,13","D 0,1->10,11",0.376280,0.162946,2.13e-01,170,1
7,N2 1.8 Å,sym,CISD,185.15,"D 8,9->12,13","D 8,9->18,19",0.370993,0.142491,2.29e-01,170,1
8,N2 3 Å,sym,HF,517.13,"D 8,9->10,11","D 8,9->10,21",0.470748,0.176788,2.94e-01,164,1
9,N2 3 Å,sym,CISD,235.97,"D 2,3->14,15","D 4,5->18,19",0.078666,0.078025,6.41e-04,166,2


## 5. Exact-winner costs (Part I definition)
Oracle context-shots. M3C = contrast rule on the parent groups.

In [14]:
methods(df)

,Case,Pool,State,M1,M2,M3,M3R,M3C,M1/M2,M2/M3,M2/M3R,M3/M3C
0,H2O 0.9584 Å,sym,HF,2.71e+07,7.22e+06,4.91e+06,4.91e+06,2.14e+06,3.75,1.47,1.47,2.29
1,H2O 0.9584 Å,sym,CISD,7.71e+14,3.17e+13,6.76e+13,6.76e+13,2.88e+13,24.36,0.47,0.47,2.35
2,H2O 1.75 Å,sym,HF,3.43e+07,1.86e+06,2.75e+06,1.54e+06,1.29e+06,18.45,0.68,1.20,2.13
3,H2O 1.75 Å,sym,CISD,4.03e+08,4.04e+07,3.87e+07,2.18e+07,1.80e+07,9.97,1.04,1.85,2.15
4,N2 1.1 Å,sym,HF,1.84e+20,2.45e+18,4.68e+18,1.76e+18,2.27e+18,74.94,0.52,1.39,2.06
5,N2 1.1 Å,sym,CISD,1.77e+22,2.71e+20,5.46e+20,1.92e+20,2.66e+20,65.42,0.50,1.41,2.05
6,N2 1.8 Å,sym,HF,4.90e+05,1.29e+06,2.61e+05,2.60e+05,7.40e+04,0.38,4.92,4.95,3.53
7,N2 1.8 Å,sym,CISD,4.05e+05,7.22e+05,2.39e+05,2.39e+05,7.26e+04,0.56,3.03,3.03,3.29
8,N2 3 Å,sym,HF,2.94e+05,7.43e+05,1.69e+05,1.69e+05,4.70e+04,0.40,4.39,4.39,3.60
9,N2 3 Å,sym,CISD,5.10e+10,3.09e+09,5.24e+09,2.44e+09,2.61e+09,16.50,0.59,1.27,2.01


## 6. Δ-good costs (Δ = 1 mHa)

In [15]:
methods(df, delta=True)

,Case,Pool,State,M1,M2,M3,M3R,M3C,M1/M2,M2/M3,M2/M3R,M3/M3C
0,H2O 0.9584 Å,sym,HF,2.71e+07,7.22e+06,4.91e+06,4.91e+06,2.14e+06,3.75,1.47,1.47,2.29
1,H2O 0.9584 Å,sym,CISD,2.98e+10,4.17e+10,1.80e+10,1.80e+10,6.37e+09,0.71,2.32,2.32,2.83
2,H2O 1.75 Å,sym,HF,3.43e+07,1.86e+06,2.75e+06,1.54e+06,1.29e+06,18.45,0.68,1.20,2.13
3,H2O 1.75 Å,sym,CISD,4.03e+08,4.04e+07,3.87e+07,2.18e+07,1.80e+07,9.97,1.04,1.85,2.15
4,N2 1.1 Å,sym,HF,3.70e+10,4.96e+08,9.41e+08,3.55e+08,4.56e+08,74.50,0.53,1.40,2.07
5,N2 1.1 Å,sym,CISD,3.65e+10,1.05e+09,1.19e+09,5.50e+08,5.66e+08,34.81,0.88,1.91,2.10
6,N2 1.8 Å,sym,HF,4.90e+05,1.29e+06,2.61e+05,2.60e+05,7.40e+04,0.38,4.92,4.95,3.53
7,N2 1.8 Å,sym,CISD,4.05e+05,7.22e+05,2.39e+05,2.39e+05,7.26e+04,0.56,3.03,3.03,3.29
8,N2 3 Å,sym,HF,2.94e+05,7.43e+05,1.69e+05,1.69e+05,4.70e+04,0.40,4.39,4.39,3.60
9,N2 3 Å,sym,CISD,2.10e+10,1.31e+09,2.16e+09,1.03e+09,1.08e+09,16.03,0.60,1.27,2.01


## 7. Effect of symmetry filtering on H₂O (full / sym cost ratio)

In [16]:
h = df[df.system == "H2O"]
f = h[h.pool_type == "full"].set_index(["case", "state"]); s = h[h.pool_type == "sym"].set_index(["case", "state"])
cols = ["M1", "M2", "M3", "M1_Delta", "M2_Delta", "M3_Delta", "M3C_Delta"]
(f[cols] / s[cols]).round(2)

M1    M2    M3  M1_Delta  M2_Delta  M3_Delta  M3C_Delta
case         state                                                           
H2O 0.9584 Å HF     3.22  2.14  1.95      3.22      2.14      1.95       1.79
             CISD   3.22  1.18  1.50      3.22      2.32      2.31       2.24
H2O 1.75 Å   HF     3.68  2.52  1.82      3.68      2.52      1.82       1.68
             CISD   3.70  2.95  2.14      3.70      2.95      2.14       1.93